# **MLProcess - Smoke Detector**
---
**3 - Data Preprocessing**

In [1]:
# Import the required libraries.
import pandas as pd

from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

import src.utils as utils

**Summary:**
1. There are no missing values.
2. There are no categorical features.
3. The are exists outliers.
4. The labels are already encoded.
5. The labels are highly imbalanced (71.5% class 1).

## **1 - Configuration File**
---

In [2]:
# Load the conifguration file.
config = utils.load_config()
utils.print_debug("Config file loaded...")

2026-03-07 02:16:38.150917 Config file loaded...


In [3]:
# Check the configuration file.
config

{'columns_datetime': ['utc'],
 'columns_float': ['temperature',
  'humidity_pct',
  'pressure',
  'pm10',
  'pm25',
  'nc05',
  'nc10',
  'nc25'],
 'columns_int': ['tvoc', 'co2', 'raw_h2', 'raw_ethanol', 'fire_alarm'],
 'features': ['temperature',
  'humidity_pct',
  'pressure',
  'pm10',
  'tvoc',
  'co2',
  'raw_h2',
  'raw_ethanol'],
 'label': 'fire_alarm',
 'path_data_raw': 'data/raw/smoke_detection_iot.csv',
 'path_data_test': ['data/interim/X_test.pkl', 'data/interim/y_test.pkl'],
 'path_data_train': ['data/interim/X_train.pkl', 'data/interim/y_train.pkl'],
 'path_data_valid': ['data/interim/X_valid.pkl', 'data/interim/y_valid.pkl'],
 'path_data_validated': 'data/interim/validated_data.pkl',
 'range_co2': [400, 60000],
 'range_fire_alarm': [0, 1],
 'range_humidity_pct': [10.74, 75.2],
 'range_nc05': [0, 65535],
 'range_nc10': [0, 65535],
 'range_nc25': [0, 65535],
 'range_pm10': [0.0, 14333.69],
 'range_pm25': [0, 65535],
 'range_pressure': [930.852, 939.861],
 'range_raw_ethanol

## **2 - Load Data**
---

In [4]:
# Function for load data.
def load_data(config):
    """
    Load every set of data.

    Parameters:
    ----------
    config : dict
        The loaded configuration file.

    Returns:
    -------
    data_train, data_valid, data_test : pd.DataFrame
        The loaded data.
    """

    # Load the train set.
    X_train = utils.deserialize_data(config["path_data_train"][0])
    y_train = utils.deserialize_data(config["path_data_train"][1])

    # Load the valid set.
    X_valid = utils.deserialize_data(config["path_data_valid"][0])
    y_valid = utils.deserialize_data(config["path_data_valid"][1])

    # Load the test set.
    X_test = utils.deserialize_data(config["path_data_test"][0])
    y_test = utils.deserialize_data(config["path_data_test"][1])

    # Concatenate the X and y of each set.
    data_train = pd.concat([X_train, y_train], axis=1)
    data_valid = pd.concat([X_valid, y_valid], axis=1)
    data_test = pd.concat([X_test, y_test], axis=1)

    # Validate the proportion.
    num_all_data = int(data_train.shape[0]) + int(data_valid.shape[0]) + int(data_test.shape[0])
    print(f"Data train proportion : {len(X_train) / num_all_data}")
    print(f"Data valid proportion : {len(X_valid) / num_all_data}")
    print(f"Data test proportion  : {len(X_test) / num_all_data}")

    return data_train, data_valid, data_test

In [5]:
# Load the data.
data_train, data_valid, data_test = load_data(config)

Data deserialized from data/interim/X_train.pkl
Data deserialized from data/interim/y_train.pkl
Data deserialized from data/interim/X_valid.pkl
Data deserialized from data/interim/y_valid.pkl
Data deserialized from data/interim/X_test.pkl
Data deserialized from data/interim/y_test.pkl
Data train proportion : 0.8
Data valid proportion : 0.1
Data test proportion  : 0.1


In [6]:
# Sanity check the train data.
data_train

,temperature,humidity_pct,pressure,pm10,tvoc,co2,raw_h2,raw_ethanol,fire_alarm
51420,29.520,39.97,937.567,1.86,26,400,12848,20752,0
56406,59.890,11.23,936.761,463.79,60000,3818,11846,17318,0
52217,27.140,42.63,937.511,2.19,117,433,12770,20621,0
55564,49.950,24.31,936.897,0.73,21060,653,12679,18931,0
14753,14.112,53.18,938.854,2.06,1141,456,12858,19446,1
...,...,...,...,...,...,...,...,...,...
61185,20.514,29.66,936.873,0.81,5724,400,13043,19602,0
30715,22.040,49.19,939.656,2.55,32,400,13258,20228,1
41785,24.570,53.34,938.769,1.69,1166,402,12896,19448,1
47379,26.290,50.96,938.731,1.80,1310,400,12970,19404,1


## **3 - Outliers Removal**
---

We will implement IQR-based outliers removal.

In [7]:
# Function for outliers removal.
def remove_outliers(data):
    """
    IQR-based outliers removal.

    Parameters:
    ----------
    data : pd.DataFrame
        Loaded data.

    Returns:
    -------
    data_clean : pd.DataFrame
        Cleaned data.
    """

    # Ensure raw data immutable.
    data = data.copy()
    list_cleaned_data = []

    # For each features (exclude label).
    for col in data.columns[:-1]:
        q1 = data[col].quantile(0.25)
        q3 = data[col].quantile(0.75)
        iqr = q3 - q1

        # Boolean condition for outliers.
        q1_cond = data[col] < (q1 - (1.5 * iqr))
        q3_cond = data[col] > (q3 + (1.5 * iqr))

        clean = data[~(q1_cond | q3_cond)]
        list_cleaned_data.append(clean)

    data_clean = pd.concat(list_cleaned_data)
    n_duplicated_idx = data_clean.index.value_counts()
    used_idx = n_duplicated_idx[n_duplicated_idx == (data.shape[1]-1)].index
    data_clean = data_clean.loc[used_idx].drop_duplicates()

    print(f"Original data shape    : {data.shape}")
    print(f"Dropped data shape     : {data_clean.shape}")
    print(f"Number of dropped data : {data.shape[0] - data_clean.shape[0]}")

    return data_clean

In [8]:
# Remove the outliers.
data_train_clean = remove_outliers(data_train)

Original data shape    : (50104, 9)
Dropped data shape     : (29961, 9)
Number of dropped data : 20143


## **4 - Scaling Data**
---

Standardize the data using `StandardScaler`

In [9]:
# Function to fit the scaler.
def fit_scaler(data, path_scaler):
    """
    Fit the scaler.
    
    Parameters:
    ----------
    data : pd.DataFrame
        Input data (all features must be in numeric form)

    path_scaler : str
        The scaler location.
        
    Returns:
    -------
    scaler : sklearn.preprocessing.StandardScaler
        Fitted scaler object (storing the mean & std of all features)
    """

    # Split input-output, StandardScaler() only accepts numeric data.
    label = config["label"]
    y = data[label]
    X = data.drop(columns = label)
    
    # Create scaler object.
    scaler = StandardScaler()

    # Fit the scaler.
    scaler.fit(X)

    # Serialize the scaler.    
    utils.serialize_data(scaler, path_scaler)
    
    return scaler

In [10]:
# Function to scale the data.
def transform_scaler(data, scaler):
    """
    Transform the data using scaler.
    
    Parameters:
    ----------
    data : pd.DataFrame
        Input data (all features must be in numeric form)    
        
    scaler : sklearn.preprocessing.StandardScaler
        Fitted scaler object (storing the mean & std of all features)
        
    Returns:
    -------
    data : pd.DataFrame
        The scaled data
    """

    # Ensure raw data immutable.
    data = data.copy()

    # Split input-output, StandardScaler() only accepts numeric data.
    label = config["label"]
    y = data[label]
    X = data.drop(columns = label)

    # Scale the data.
    scaled_data = scaler.transform(X)

    # Convert to dataframe.
    X_scaled = pd.DataFrame(
        scaled_data,
        columns = X.columns,
        index = X.index
    )

    # Concat the X_scaled with y.
    data = pd.concat(
        [X_scaled, y],
        axis = 1
    )
    
    return data

In [11]:
# Fit the scaler.
PATH_SCALER = "models/scaler.pkl"

scaler = fit_scaler(
    data = data_train_clean,
    path_scaler = PATH_SCALER
)

# Update the configuration file.
config = utils.update_config(
    key = "path_fitted_scaler",
    value = PATH_SCALER,
    config = config
)

Data serialized to models/scaler.pkl
Params Updated! 
Key: path_fitted_scaler 
Value: models/scaler.pkl



- Scale the data in train, valid, and test set.

In [12]:
data_train = transform_scaler(
    data = data_train_clean,
    scaler = scaler
)

print(f"Data shape : {data_train.shape}")
data_train.head()

Data shape : (29961, 9)


,temperature,humidity_pct,pressure,pm10,tvoc,co2,raw_h2,raw_ethanol,fire_alarm
39483,1.002946,-1.528159,-0.557151,0.976230,0.859642,4.103077,-1.240708,-0.874186,1
47379,0.926177,-0.133637,-0.983044,0.488360,1.109081,-0.445069,-0.379715,-0.983040,1
41785,0.772640,0.681833,-0.896034,0.330520,0.844969,-0.331365,-0.919660,-0.842170,1
30715,0.546797,-0.740100,1.134973,1.564542,-1.234917,-0.445069,1.721690,1.655083,1
3808,-1.269406,-1.678918,1.155581,0.732295,-1.009321,-0.445069,1.378752,1.120415,1


In [13]:
# Scale the data.
data_valid = transform_scaler(
    data = data_valid,
    scaler = scaler
)

print(f"Data shape : {data_valid.shape}")
data_valid.head()

Data shape : (6263, 9)


,temperature,humidity_pct,pressure,pm10,tvoc,co2,raw_h2,raw_ethanol,fire_alarm
28060,0.399509,-0.150769,1.139552,-0.745661,-1.040501,-0.445069,1.116076,1.024367,0
23346,-3.345462,-0.270691,-1.054026,0.316171,1.158602,-0.445069,-0.350529,-1.034266,1
27168,0.113858,0.952514,1.050252,-0.803058,-1.293609,-0.445069,1.210932,1.315713,0
11986,0.063869,0.445415,-0.273222,0.373568,0.729419,11.323259,-1.517976,-0.835766,1
16311,-0.307566,0.650996,-0.939539,0.057888,0.828462,0.407708,-1.029108,-0.880589,1


In [14]:
# Scale the data.
data_test = transform_scaler(
    data = data_test,
    scaler = scaler
)

print(f"Data shape : {data_test.shape}")
data_test.head()

Data shape : (6263, 9)


,temperature,humidity_pct,pressure,pm10,tvoc,co2,raw_h2,raw_ethanol,fire_alarm
16413,-0.290606,-0.880580,-0.921221,0.789691,0.844969,0.919375,-1.087480,-0.874186,1
38673,0.764606,0.685259,-0.664769,1.076673,0.747761,5.979188,-1.269894,-0.819758,1
25918,-0.050391,0.346051,1.352499,-1.664003,-1.288107,-0.445069,-0.014888,0.713811,0
24241,-2.202323,0.777771,-0.873136,1.406702,0.782609,-0.445069,0.065374,-0.848573,1
22626,-3.204332,-0.304954,-0.914352,0.603153,1.028380,-0.445069,-0.314046,-0.979839,1


## **5 - Label Balancing**
---

Label balancing using SMOTE.

In [15]:
# Check the label distribution.
data_train["fire_alarm"].value_counts()

fire_alarm
1    25232
0     4729
Name: count, dtype: int64

In [16]:
# Function to balancing the label.
def label_balancer(data, config, random_state=123):
    """
    Balancing the category label.

    Parameters:
    ----------
    data : pd.DataFrame
        Loaded train set.

    config : dict
        The loaded configuration file.

    random_state : int, default = 123
        For reproducibility.

    Returns:
    -------
    X_balanced : pd.DataFrame
        The features with balanced label.

    y_balanced : pd.Series
        The label with balanced label.
    """

    # Ensure raw data immutable.
    data = data.copy()

    # Split input-output, imblearn-style similar to sklearn-style.
    label = config["label"]
    y = data[label]
    X = data.drop(columns = label)

    # Set the balancer.
    balancer = SMOTE(random_state = random_state)

    # Fit resample the balancer.
    X_balanced, y_balanced = balancer.fit_resample(X, y)

    # Convert to pandas format.
    X_balanced = pd.DataFrame(X_balanced, columns=X.columns)
    y_balanced = pd.Series(y_balanced, name=label)

    print(f"The label are balanced using {balancer.__class__.__name__}")

    # Check the label distribution.
    print(y_balanced.value_counts())

    return X_balanced, y_balanced

In [17]:
# Label balancing.
X_sm, y_sm = label_balancer(
    data = data_train,
    config = config
)

The label are balanced using SMOTE
fire_alarm
1    25232
0    25232
Name: count, dtype: int64


## **6 - Data Serialization**
---

In [18]:
# Define data configuration.
label = config["label"]

y_valid = data_valid[label]
X_valid = data_valid.drop(columns = label)

y_test = data_valid[label]
X_test = data_test.drop(columns = label)

data_configuration = {
    "train": {
        "X_train": X_sm,
        "y_train": y_sm
    },
    "valid": {
        "X_valid": X_valid,
        "y_valid": y_valid
    },
    "test": {
        "X_test": X_test,
        "y_test": y_test
    }
}

In [19]:
# Serialize the preprocessed data.
PATH_PROCESSED_DATA = "data/processed/"

for key, value in data_configuration.items():
    config_key = f"path_clean_{key}"
    config_value = []

    for v in value:
        # Get each path.
        path = f"{PATH_PROCESSED_DATA + v}_clean.pkl"
        config_value.append(path)

        # Get each data.
        data = value[v]

        # Serialize the preprocessed data.
        utils.serialize_data(data, path)

    # Update the configuration parameters.
    config = utils.update_config(
        key = config_key,
        value = config_value,
        config = config
    )

Data serialized to data/processed/X_train_clean.pkl
Data serialized to data/processed/y_train_clean.pkl
Params Updated! 
Key: path_clean_train 
Value: ['data/processed/X_train_clean.pkl', 'data/processed/y_train_clean.pkl']

Data serialized to data/processed/X_valid_clean.pkl
Data serialized to data/processed/y_valid_clean.pkl
Params Updated! 
Key: path_clean_valid 
Value: ['data/processed/X_valid_clean.pkl', 'data/processed/y_valid_clean.pkl']

Data serialized to data/processed/X_test_clean.pkl
Data serialized to data/processed/y_test_clean.pkl
Params Updated! 
Key: path_clean_test 
Value: ['data/processed/X_test_clean.pkl', 'data/processed/y_test_clean.pkl']



In [20]:
# Check the configuration file.
config

{'columns_datetime': ['utc'],
 'columns_float': ['temperature',
  'humidity_pct',
  'pressure',
  'pm10',
  'pm25',
  'nc05',
  'nc10',
  'nc25'],
 'columns_int': ['tvoc', 'co2', 'raw_h2', 'raw_ethanol', 'fire_alarm'],
 'features': ['temperature',
  'humidity_pct',
  'pressure',
  'pm10',
  'tvoc',
  'co2',
  'raw_h2',
  'raw_ethanol'],
 'label': 'fire_alarm',
 'path_clean_test': ['data/processed/X_test_clean.pkl',
  'data/processed/y_test_clean.pkl'],
 'path_clean_train': ['data/processed/X_train_clean.pkl',
  'data/processed/y_train_clean.pkl'],
 'path_clean_valid': ['data/processed/X_valid_clean.pkl',
  'data/processed/y_valid_clean.pkl'],
 'path_data_raw': 'data/raw/smoke_detection_iot.csv',
 'path_data_test': ['data/interim/X_test.pkl', 'data/interim/y_test.pkl'],
 'path_data_train': ['data/interim/X_train.pkl', 'data/interim/y_train.pkl'],
 'path_data_valid': ['data/interim/X_valid.pkl', 'data/interim/y_valid.pkl'],
 'path_data_validated': 'data/interim/validated_data.pkl',
 'pat